In [0]:
from pyspark.sql import functions as F

RT = "/Volumes/transit/bronze/landing/rt/vehicle_positions"
raw = spark.read.option("multiLine", True).json(RT)

raw.printSchema()
print(raw.count(), "files")

In [0]:
v = (raw
  .select(
      F.col("_metadata.file_path").alias("_source_file"),
      F.col("header.timestamp").alias("feed_ts"),
      F.explode("entity").alias("e"))
  .select(
      "_source_file", "feed_ts",
      F.col("e.id").alias("entity_id"),
      F.col("e.vehicle.vehicle.id").alias("vehicle_id"),
      F.col("e.vehicle.trip.route_id").alias("route_id"),
      F.col("e.vehicle.trip.direction_id").alias("direction_id"),
      F.col("e.vehicle.current_status").alias("current_status"),
      F.col("e.vehicle.timestamp").alias("vehicle_ts"),
      F.col("e.vehicle.position.latitude").alias("lat"),
      F.col("e.vehicle.position.longitude").alias("lon"),
      F.col("e.vehicle.position.bearing").alias("bearing"),
      F.col("e.vehicle.occupancy_percentage").alias("occupancy_pct"),
      F.col("e.vehicle.current_stop_sequence").alias("stop_seq"))
)
v.createOrReplaceTempView("v")
display(v.limit(20))

filter, group, aggregate


In [0]:
%sql
SELECT route_id, count(*) AS rows, count(DISTINCT vehicle_id) AS vehicles
FROM v
WHERE route_id IS NOT NULL
GROUP BY route_id
HAVING count(*) > 50
ORDER BY rows DESC

In [0]:
(v.filter(F.col("route_id").isNotNull())
  .groupBy("route_id")
  .agg(F.count("*").alias("rows"),
       F.countDistinct("vehicle_id").alias("vehicles"))
  .filter(F.col("rows") > 50)
  .orderBy(F.desc("rows"))
  .display())

Join

In [0]:
route_types = spark.createDataFrame(
    [("Red", 1), ("Orange", 1), ("Blue", 1), ("Green-B", 0), ("Green-C", 0),
     ("Green-D", 0), ("Green-E", 0), ("Mattapan", 0)],
    "route_id string, route_type int")
route_types.createOrReplaceTempView("route_types")

Case


In [0]:
crowding = (F.when(F.col("occupancy_pct").isNull(), "unknown")
             .when(F.col("occupancy_pct") < 30, "light")
             .when(F.col("occupancy_pct") < 70, "moderate")
             .otherwise("crowded"))

(v.withColumn("crowding", crowding)
  .groupBy("current_status", "crowding")
  .count()
  .orderBy("current_status", "crowding")
  .display())

window function

In [0]:
%sql
SELECT vehicle_id, route_id, vehicle_ts,
       row_number() OVER (PARTITION BY route_id ORDER BY vehicle_ts DESC) AS rn,
       lag(vehicle_ts) OVER (PARTITION BY vehicle_id ORDER BY vehicle_ts) AS prev_ts,
       vehicle_ts - lag(vehicle_ts) OVER (PARTITION BY vehicle_id ORDER BY vehicle_ts) AS gap_s
FROM v WHERE route_id IS NOT NULL
ORDER BY vehicle_id, vehicle_ts
LIMIT 100

Dedup


In [0]:
%sql
SELECT vehicle_id, route_id, vehicle_ts, lat, lon
FROM v
WHERE vehicle_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY vehicle_id ORDER BY vehicle_ts DESC) = 1

the second explode

In [0]:
carriages = (raw
  .select(F.explode("entity").alias("e"))
  .select(
      F.col("e.vehicle.vehicle.id").alias("vehicle_id"),
      F.col("e.vehicle.timestamp").alias("vehicle_ts"),
      F.col("e.vehicle.multi_carriage_details").alias("carriages"))
  .filter(F.col("carriages").isNotNull())
  .select("vehicle_id", "vehicle_ts",
          F.posexplode("carriages").alias("carriage_pos", "c"))
  .select("vehicle_id", "vehicle_ts", "carriage_pos",
          F.col("c.label").alias("carriage_label"),
          F.col("c.carriage_sequence").alias("carriage_sequence"),
          F.col("c.occupancy_status").alias("carriage_occupancy_status"),
          F.col("c.occupancy_percentage").alias("carriage_occupancy_pct"),
          F.col("c.orientation").alias("carriage_orientation")))

print(v.count(), "vehicle rows →", carriages.count(), "carriage rows")
display(carriages.limit(30))

In [0]:
for f in v.schema.fields:
    print(f"{f.name:28} {f.dataType.simpleString():12} nullable={f.nullable}")

In [0]:
from pyspark.sql.types import StructType, ArrayType

def paths(schema, prefix=""):
    for f in schema.fields:
        p = f"{prefix}{f.name}"
        dt = f.dataType
        if isinstance(dt, StructType):
            yield from paths(dt, p + ".")
        elif isinstance(dt, ArrayType) and isinstance(dt.elementType, StructType):
            yield f"{p}[]"
            yield from paths(dt.elementType, p + "[].")
        else:
            yield f"{p}  ({dt.simpleString()})"

for p in paths(raw.schema):
    print(p)

In [0]:

import time

t0 = time.time()
chain = (v.filter(F.col("route_id").isNotNull())
          .withColumn("crowding",
                      F.when(F.col("occupancy_pct") > 50, "high").otherwise("low"))
          .groupBy("route_id", "crowding")
          .agg(F.count("*").alias("n"))
          .orderBy(F.desc("n")))
print(f"building the chain: {time.time()-t0:.3f}s")

t0 = time.time()
result = chain.collect()
print(f"collecting:         {time.time()-t0:.3f}s, {len(result)} rows")

In [0]:
v.select(F.col("lat"), F.col("lon"),
         (F.col("occupancy_pct") / 100).alias("occ_frac")).limit(5).display()

v.selectExpr("lat", "lon", "occupancy_pct / 100 AS occ_frac").limit(5).display()

v.select("lat", "lon", F.expr("occupancy_pct / 100 AS occ_frac")).limit(5).display()

In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, LongType, IntegerType, ArrayType)

position_schema = StructType([
    StructField("latitude",  DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("bearing",   DoubleType()),
])

trip_schema = StructType([
    StructField("route_id",     StringType()),
    StructField("trip_id",      StringType()),
    StructField("direction_id", IntegerType()),
])

vehicle_schema = StructType([
    StructField("current_status",        StringType()),
    StructField("timestamp",             LongType()),
    StructField("current_stop_sequence", IntegerType()),
    StructField("occupancy_percentage",  IntegerType()),
    StructField("position",              position_schema),
    StructField("trip",                  trip_schema),
    StructField("vehicle", StructType([
        StructField("id",    StringType()),
        StructField("label", StringType()),
    ])),
])

feed_schema = StructType([
    StructField("header", StructType([StructField("timestamp", LongType())])),
    StructField("entity", ArrayType(StructType([
        StructField("id",      StringType()),
        StructField("vehicle", vehicle_schema),
    ]))),
])

In [0]:
v.select(F.col("lat"), F.col("lon"),
         (F.col("occupancy_pct") / 100).alias("occ_frac")).limit(5).display()

v.selectExpr("lat", "lon", "occupancy_pct / 100 AS occ_frac").limit(5).display()

v.select("lat", "lon", F.expr("occupancy_pct / 100 AS occ_frac")).limit(5).display()

In [0]:
%sql
SELECT 'gtfs_stops' AS t, count(*) AS rows FROM transit.bronze.gtfs_stops
UNION ALL SELECT 'gtfs_routes',         count(*) FROM transit.bronze.gtfs_routes
UNION ALL SELECT 'gtfs_trips',          count(*) FROM transit.bronze.gtfs_trips
UNION ALL SELECT 'gtfs_stop_times',     count(*) FROM transit.bronze.gtfs_stop_times
UNION ALL SELECT 'gtfs_calendar',       count(*) FROM transit.bronze.gtfs_calendar
UNION ALL SELECT 'gtfs_calendar_dates', count(*) FROM transit.bronze.gtfs_calendar_dates
ORDER BY rows DESC